# L0 vs F1 curves — hidden_200_l8 runs

Compares SAE architectures trained on the `hidden_200` benchmark with integer L0 targets 2–11
(`d_in=200`, `d_sae=648`, 60 M training samples).

Data sources:
- `data/hidden_200_l8_standard/` — Standard (ReLU)
- `data/hidden_200_l8_batch/` — BatchTopK
- `data/hidden_200_l8_matryoshka/` — Matryoshka
- `data/hidden_200_l8_mp/` — Matching Pursuit (pending)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from occhio.benchmark import benchmark_ae_baselines

# ── Data loading ──────────────────────────────────────────────────────────────
DATA_FOLDERS = {
    "Standard": "data/hidden_200_l8_standard/results.parquet",
    "BatchTopK": "data/hidden_200_l8_batch/results.parquet",
    "Matryoshka": "data/hidden_200_l8_matryoshka/results.parquet",
    "MatchingPursuit": "data/hidden_200_l8_mp/results.parquet",
}

dfs = []
for label, path in DATA_FOLDERS.items():
    try:
        d = pd.read_parquet(path).reset_index()
        dfs.append(d)
    except FileNotFoundError:
        print(f"Skipping {label}: {path} not found")

raw = pd.concat(dfs, ignore_index=True)
raw = raw.rename(columns={"f1_score": "f1"})

best = (
    raw.groupby(["benchmark", "sae_type", "sae_l0"], as_index=False)
    .agg(
        f1=("f1", "max"),
        mcc=("mcc", "max"),
        precision=("precision", "max"),
        recall=("recall", "max"),
    )
    .sort_values(["benchmark", "sae_type", "sae_l0"])
    .reset_index(drop=True)
)

true_l0_per_benchmark = raw.groupby("benchmark")["true_l0"].first().to_dict()

# ── AE baseline (HuggingFace AE, n_hidden=200) ───────────────────────────────
ae_baselines = benchmark_ae_baselines(
    device="mps",
    cache_path="data/ae_baselines_hidden_200_l8_huggingface.parquet",
    ae_type="huggingface",
    ae_kwargs={"n_hidden": 200},
    n_eval_features=648,
)

print(f"Benchmarks:    {sorted(best['benchmark'].unique())}")
print(f"Architectures: {best['sae_type'].unique().tolist()}")
print(f"Rows:          {len(best)}")
print(f"\nTrue L0 per benchmark:\n{true_l0_per_benchmark}")
print(f"\nAE baselines:\n{ae_baselines}")

In [ ]:
# ── Paper-ready style ─────────────────────────────────────────────────────────
PAPER_COLORS = {
    "BatchTopK": "#000C7A",
    "Matryoshka": "#DC2626",
    "MatchingPursuit": "#297A58",
    "Standard": "#FCBA03",
}
PAPER_MARKERS = {
    "Standard": "circle",
    "BatchTopK": "square",
    "Matryoshka": "diamond",
    "MatchingPursuit": "triangle-up",
}
PAPER_COLOR_AE = "#000000"
PAPER_COLOR_TRUE_L0 = "#0891B2"

_FONT = dict(family="Times New Roman, serif", size=28, color="#333333")
_AXIS_STYLE = dict(
    showgrid=False,
    zeroline=False,
    linecolor="#666666",
    linewidth=1,
    mirror=True,
    ticks="outside",
    ticklen=4,
    tickwidth=1,
    tickcolor="#666666",
    tickfont_size=18,
)
_LINE_WIDTH = 2.5
_SUBPLOT_TITLE_FONT = dict(family="Times New Roman, serif", size=22, color="#333333")
_AXIS_TITLE_FONT = dict(family="Times New Roman, serif", size=30, color="#333333")

BENCHMARK_LABELS = {
    "CORRELATED_PAIRS": "Correlated Pairs",
    "DAG_RANDOM_WALK": "Deep Hierarchy",
    "HIERARCHICAL_PAIRS": "Hierarchical Pairs",
    "POWER_LAW_DIGRAPH": "Preferential Attachment",
    "SIMPLICIAL_COMPLEX": "Simplicial Complex",
    "SPARSE_UNIFORM": "Zipfian",
    "SPHERICAL": "Spherical",
    "TORUS": "Torus",
}

BENCHMARK_ORDER = [
    "SPARSE_UNIFORM",
    "CORRELATED_PAIRS",
    "HIERARCHICAL_PAIRS",
    "POWER_LAW_DIGRAPH",
    "DAG_RANDOM_WALK",
    "SIMPLICIAL_COMPLEX",
    "SPHERICAL",
    "TORUS",
]

ARCH_LABELS = {
    "BatchTopK": r"$\text{BatchTopK}$",
    "Matryoshka": r"$\text{Matryoshka}$",
    "MatchingPursuit": r"$\text{Matching Pursuit}$",
    "Standard": r"$\text{ReLU}$",
}

PAPER_ARCH_ORDER = ["Standard", "BatchTopK", "Matryoshka", "MatchingPursuit"]

_LABEL_TRUE_L0 = r"$L^0_{\text{True}}$"
_LABEL_AE_BASELINE = r"$\text{AE baseline}$"


def make_l0_f1_paper() -> go.Figure:
    n_cols, n_rows = 4, 2
    subplot_titles = [BENCHMARK_LABELS.get(b, b) for b in BENCHMARK_ORDER]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_xaxes=False,
        shared_yaxes=False,
        horizontal_spacing=0.02,
        vertical_spacing=0.12,
    )

    x_max = int(best["sae_l0"].max())
    x_padding = 0.3

    shown_legends: set[str] = set()
    baseline_label_data: list = []

    for idx, benchmark in enumerate(BENCHMARK_ORDER):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        sub = best[best["benchmark"] == benchmark]

        baseline_key = benchmark.lower()

        # AE baseline horizontal dashed line (drawn first so SAE traces sit on top)
        if baseline_key in ae_baselines.index:
            baseline_val = ae_baselines.loc[baseline_key, "precision_macro"]
            show_bl = _LABEL_AE_BASELINE not in shown_legends
            shown_legends.add(_LABEL_AE_BASELINE)
            fig.add_trace(
                go.Scatter(
                    x=[1 - x_padding, x_max + x_padding],
                    y=[baseline_val, baseline_val],
                    mode="lines",
                    name=_LABEL_AE_BASELINE,
                    line=dict(color=PAPER_COLOR_AE, dash="dot", width=1),
                    legendgroup=_LABEL_AE_BASELINE,
                    showlegend=show_bl,
                ),
                row=row,
                col=col,
            )
            baseline_label_data.append((idx, baseline_val))

        # True L0 vertical line
        if benchmark in true_l0_per_benchmark:
            tl0 = true_l0_per_benchmark[benchmark]
            show_tl0 = _LABEL_TRUE_L0 not in shown_legends
            shown_legends.add(_LABEL_TRUE_L0)
            fig.add_trace(
                go.Scatter(
                    x=[tl0, tl0],
                    y=[0, 1],
                    mode="lines",
                    name=_LABEL_TRUE_L0,
                    line=dict(color=PAPER_COLOR_TRUE_L0, width=1.5, dash="dashdot"),
                    legendgroup=_LABEL_TRUE_L0,
                    showlegend=show_tl0,
                ),
                row=row,
                col=col,
            )

        for arch in PAPER_ARCH_ORDER:
            color = PAPER_COLORS[arch]
            label = ARCH_LABELS.get(arch, arch)

            arch_data = sub[sub["sae_type"] == arch].sort_values("sae_l0")
            if arch_data.empty:
                continue
            show_legend = label not in shown_legends
            shown_legends.add(label)
            fig.add_trace(
                go.Scatter(
                    x=arch_data["sae_l0"],
                    y=arch_data["f1"],
                    mode="lines+markers",
                    name=label,
                    line=dict(color=color, width=_LINE_WIDTH),
                    marker=dict(size=8, symbol=PAPER_MARKERS[arch], color=color),
                    legendgroup=label,
                    showlegend=show_legend,
                ),
                row=row,
                col=col,
            )

        fig.update_xaxes(
            range=[1 - x_padding, x_max + x_padding],
            tickmode="linear",
            tick0=1,
            dtick=1,
            minor=dict(ticks=""),
            showticklabels=row == n_rows,
            row=row,
            col=col,
            **_AXIS_STYLE,
        )
        fig.update_yaxes(
            range=[0, 1],
            minor=dict(ticks=""),
            showticklabels=col == 1,
            row=row,
            col=col,
            **_AXIS_STYLE,
        )

    fig.update_annotations(font=_SUBPLOT_TITLE_FONT)

    # AE baseline value labels at right edge
    for _idx, _bval in baseline_label_data:
        fig.add_annotation(
            text=f"{_bval:.2f}",
            x=x_max + x_padding,
            y=_bval,
            xref=f"x{_idx + 1}",
            yref=f"y{_idx + 1}",
            xanchor="right",
            yanchor="top",
            showarrow=False,
            font=dict(size=14, family="Times New Roman, serif", color=PAPER_COLOR_AE),
        )

    # Shared axis labels
    fig.add_annotation(
        text="<i>L</i><sup>0</sup><sub>SAE</sub>",
        x=0.5,
        xref="paper",
        y=-0.15,
        yref="paper",
        showarrow=False,
        font=_AXIS_TITLE_FONT,
    )
    fig.add_annotation(
        text="<i>F</i><sub>1</sub>",
        x=-0.06,
        xref="paper",
        y=0.5,
        yref="paper",
        showarrow=False,
        textangle=-90,
        font=_AXIS_TITLE_FONT,
    )

    fig.update_layout(
        template="plotly_white",
        font=_FONT,
        height=750,
        width=1400,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=90, r=8, t=120, b=95),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.12,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cccccc",
            borderwidth=1,
            font_size=20,
        ),
    )
    return fig

In [ ]:
fig_f1_paper = make_l0_f1_paper()
fig_f1_paper.show()

In [ ]:
fig_f1_paper.write_image("fig_paper_hidden_200_l8_f1.pdf")